# NB09 — Novel OG Functional Annotation

**Goal**: Determine what the 50 novel plant-enriched OGs (from NB03) actually are.

Phase 1 reported 50 eggNOG ortholog groups enriched in plant-associated species
that survived phylogenetic control, but never looked up their functional descriptions.
This notebook annotates them via:

1. eggNOG mapper `Description`, `Preferred_name`, `COG_category`
2. InterProScan domain architecture (Pfam, Gene3D, PANTHER, CDD)
3. InterProScan GO terms and MetaCyc/KEGG pathways
4. Bakta product annotations
5. MGnify cross-reference (KEGG modules, BGC)

**Output**: `data/novel_og_annotations.csv`, `data/novel_og_domains.csv`

In [1]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')

DATA = '../data'
FIG  = '../figures'
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# Load Spark
spark = get_spark_session()
print('Spark session ready')

# Load the 50 novel OGs from NB03
novel_ogs = pd.read_csv(f'{DATA}/novel_plant_markers.csv')
print(f'Novel OGs to annotate: {len(novel_ogs)}')
print(novel_ogs[['og_id', 'odds_ratio', 'or_phylo_controlled', 'prev_plant', 'prev_nonplant']].head(10))

Spark session ready
Novel OGs to annotate: 50
     og_id  odds_ratio  or_phylo_controlled  prev_plant  prev_nonplant
0  COG3569    8.920921             6.007705    0.540493       0.116493
1  COG1764    9.294146             5.393882    0.822183       0.332217
2  COG5343    7.737517             5.338255    0.626761       0.178325
3  COG0654   12.663734             8.299146    0.911972       0.449970
4  COG1845   14.466098             8.735082    0.932218       0.487369
5  COG3832   10.030044             6.441485    0.879401       0.420966
6  COG3237    7.266079             4.951443    0.699824       0.242916
7  COG0843   13.676824             6.955691    0.931338       0.497931
8  COG2321    7.086053             4.452551    0.693662       0.242167
9  COG1622   12.962730             7.873446    0.926056       0.491389


## 1. eggNOG Functional Descriptions

For each novel OG, retrieve the most common `Description`, `Preferred_name`,
`COG_category`, `KEGG_ko`, `KEGG_Module`, and `EC` across all gene clusters
carrying that OG.

In [2]:
CACHE_EGGNOG = f'{DATA}/novel_og_eggnog_annotations.csv'

if os.path.exists(CACHE_EGGNOG) and os.path.getsize(CACHE_EGGNOG) > 100:
    og_eggnog = pd.read_csv(CACHE_EGGNOG)
    print(f'Loaded cached eggNOG annotations: {len(og_eggnog)} rows')
else:
    og_list = novel_ogs['og_id'].tolist()
    og_sql = "','".join(og_list)
    
    # Query eggNOG for all gene clusters carrying these OGs
    # Extract primary OG ID the same way NB03 did
    og_eggnog = spark.sql(f"""
        SELECT SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] AS og_id,
               ann.Description,
               ann.Preferred_name,
               ann.COG_category,
               ann.KEGG_ko,
               ann.KEGG_Module,
               ann.KEGG_Pathway,
               ann.EC,
               ann.GOs,
               ann.PFAMs,
               gc.gtdb_species_clade_id,
               gc.is_core
        FROM kbase_ke_pangenome.eggnog_mapper_annotations ann
        JOIN kbase_ke_pangenome.gene_cluster gc
             ON ann.query_name = gc.gene_cluster_id
        WHERE ann.eggNOG_OGs IS NOT NULL
          AND SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] IN ('{og_sql}')
    """).toPandas()
    
    og_eggnog.to_csv(CACHE_EGGNOG, index=False)
    print(f'Retrieved eggNOG annotations: {len(og_eggnog)} gene clusters across {og_eggnog["og_id"].nunique()} OGs')

print(f'\nOGs with annotations: {og_eggnog["og_id"].nunique()}')
print(f'Total gene clusters: {len(og_eggnog):,}')

Loaded cached eggNOG annotations: 1206652 rows

OGs with annotations: 50
Total gene clusters: 1,206,652


In [3]:
# Summarize: for each OG, get most common description, preferred name, COG category
def most_common(series):
    """Return most common non-empty value."""
    vals = series.dropna().replace('', np.nan).dropna()
    if len(vals) == 0:
        return ''
    return vals.value_counts().index[0]

og_summary = og_eggnog.groupby('og_id').agg(
    n_clusters=('og_id', 'size'),
    n_species=('gtdb_species_clade_id', 'nunique'),
    pct_core=('is_core', 'mean'),
    description=('Description', most_common),
    preferred_name=('Preferred_name', most_common),
    cog_category=('COG_category', most_common),
    kegg_ko=('KEGG_ko', most_common),
    kegg_module=('KEGG_Module', most_common),
    kegg_pathway=('KEGG_Pathway', most_common),
    ec_number=('EC', most_common),
    go_terms=('GOs', most_common),
    pfams=('PFAMs', most_common),
).reset_index()

# Merge with enrichment stats
og_summary = og_summary.merge(novel_ogs[['og_id', 'odds_ratio', 'or_phylo_controlled', 
                                          'prev_plant', 'prev_nonplant', 'q_value']], 
                               on='og_id', how='left')
og_summary = og_summary.sort_values('odds_ratio', ascending=False)

print('\n=== Top 20 Novel Plant-Enriched OGs with Functional Annotations ===')
for _, row in og_summary.head(20).iterrows():
    print(f"\n{row['og_id']} | OR={row['odds_ratio']:.1f} (phylo={row['or_phylo_controlled']:.1f}) | "
          f"plant={row['prev_plant']:.1%} vs non={row['prev_nonplant']:.1%}")
    print(f"  Name: {row['preferred_name']}")
    print(f"  Description: {row['description'][:100]}")
    print(f"  COG category: {row['cog_category']} | EC: {row['ec_number']} | KEGG: {row['kegg_ko']}")
    print(f"  Species: {row['n_species']:,} | Clusters: {row['n_clusters']:,} | Core: {row['pct_core']:.1%}")


=== Top 20 Novel Plant-Enriched OGs with Functional Annotations ===

COG1845 | OR=14.5 (phylo=8.7) | plant=93.2% vs non=48.7%
  Name: ctaE
  Description: Cytochrome c oxidase subunit III
  COG category: C | EC: 1.9.3.1 | KEGG: ko:K02276
  Species: 14,329 | Clusters: 26,563 | Core: 78.1%

COG0316 | OR=13.9 (phylo=6.3) | plant=94.3% vs non=54.2%
  Name: iscA
  Description: Belongs to the HesB IscA family
  COG category: S | EC: - | KEGG: ko:K13628
  Species: 15,821 | Clusters: 32,229 | Core: 83.1%

COG0843 | OR=13.7 (phylo=7.0) | plant=93.1% vs non=49.8%
  Name: ctaD
  Description: Cytochrome c oxidase is the component of the respiratory chain that catalyzes the reduction of oxyge
  COG category: C | EC: 1.9.3.1 | KEGG: ko:K02274
  Species: 14,610 | Clusters: 33,679 | Core: 60.1%

COG0288 | OR=13.5 (phylo=7.9) | plant=94.5% vs non=55.9%
  Name: -
  Description: Reversible hydration of carbon dioxide
  COG category: P | EC: 4.2.1.1 | KEGG: ko:K01673
  Species: 16,166 | Clusters: 27,994 |

## 2. InterProScan Domain Architecture

For each novel OG, retrieve the most common InterPro domain hits
across Pfam, Gene3D, PANTHER, CDD, and SUPERFAMILY.

In [4]:
CACHE_IPR = f'{DATA}/novel_og_interproscan.csv'

if os.path.exists(CACHE_IPR) and os.path.getsize(CACHE_IPR) > 100:
    og_domains = pd.read_csv(CACHE_IPR)
    print(f'Loaded cached InterProScan domains: {len(og_domains)} rows')
else:
    og_list = novel_ogs['og_id'].tolist()
    og_sql = "','".join(og_list)
    
    # Join gene clusters carrying novel OGs to InterProScan domains
    # Use the same OG extraction as NB03
    og_domains = spark.sql(f"""
        SELECT SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] AS og_id,
               ipr.analysis,
               ipr.signature_acc,
               ipr.signature_desc,
               ipr.ipr_acc,
               ipr.ipr_desc,
               COUNT(*) AS n_hits
        FROM kbase_ke_pangenome.eggnog_mapper_annotations ann
        JOIN kbase_ke_pangenome.interproscan_domains ipr
             ON ann.query_name = ipr.gene_cluster_id
        WHERE ann.eggNOG_OGs IS NOT NULL
          AND SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] IN ('{og_sql}')
          AND ipr.analysis IN ('Pfam', 'Gene3D', 'PANTHER', 'CDD', 'SUPERFAMILY', 'NCBIfam')
        GROUP BY 1, 2, 3, 4, 5, 6
    """).toPandas()
    
    og_domains.to_csv(CACHE_IPR, index=False)
    print(f'Retrieved InterProScan domains: {len(og_domains)} domain-OG pairs across {og_domains["og_id"].nunique()} OGs')

print(f'\nOGs with domain hits: {og_domains["og_id"].nunique()}/50')
print(f'\nDomain database breakdown:')
print(og_domains.groupby('analysis')['og_id'].nunique().sort_values(ascending=False))

Loaded cached InterProScan domains: 4455 rows

OGs with domain hits: 50/50

Domain database breakdown:
analysis
Gene3D         50
SUPERFAMILY    50
Pfam           50
PANTHER        49
CDD            45
NCBIfam        42
Name: og_id, dtype: int64


In [5]:
# Top domains per OG (Pfam preferred, then CDD, then Gene3D)
priority = {'Pfam': 1, 'CDD': 2, 'NCBIfam': 3, 'Gene3D': 4, 'PANTHER': 5, 'SUPERFAMILY': 6}
og_domains['priority'] = og_domains['analysis'].map(priority).fillna(99)

# For each OG, get the top domain from the highest-priority database
top_domains = (og_domains
    .sort_values(['og_id', 'priority', 'n_hits'], ascending=[True, True, False])
    .groupby('og_id')
    .first()
    .reset_index())

# Merge into main summary
og_summary = og_summary.merge(
    top_domains[['og_id', 'analysis', 'signature_acc', 'signature_desc', 'ipr_acc', 'ipr_desc']]
        .rename(columns={'analysis': 'top_domain_db', 'signature_acc': 'top_domain_acc',
                         'signature_desc': 'top_domain_desc', 'ipr_acc': 'interpro_acc',
                         'ipr_desc': 'interpro_desc'}),
    on='og_id', how='left'
)

print('=== Domain Architecture for Top 20 OGs ===')
for _, row in og_summary.head(20).iterrows():
    domains_for_og = og_domains[og_domains['og_id'] == row['og_id']]
    pfam_domains = domains_for_og[domains_for_og['analysis'] == 'Pfam']
    print(f"\n{row['og_id']} — {row['preferred_name'] or row['description'][:60]}")
    if len(pfam_domains) > 0:
        for _, d in pfam_domains.head(3).iterrows():
            print(f"  Pfam: {d['signature_acc']} — {d['signature_desc']} (n={d['n_hits']:,})")
    if row.get('interpro_acc') and str(row['interpro_acc']) != 'nan' and str(row['interpro_acc']) != '':
        print(f"  InterPro: {row['interpro_acc']} — {row['interpro_desc']}")

=== Domain Architecture for Top 20 OGs ===

COG1845 — ctaE
  Pfam: PF13442 — Cytochrome C oxidase, cbb3-type, subunit III (n=10)
  Pfam: PF00034 — Cytochrome c (n=13)
  Pfam: PF00115 — Cytochrome C and Quinol oxidase polypeptide I (n=47)
  InterPro: IPR000298 — Cytochrome c oxidase subunit III-like

COG0316 — iscA
  Pfam: PF01106 — NifU-like domain (n=2,529)
  Pfam: PF01521 — Iron-sulphur cluster biosynthesis (n=29,932)
  Pfam: PF01592 — NifU-like N terminal domain (n=2)
  InterPro: IPR000361 — Core domain, A-type assembly protein ATAP

COG0843 — ctaD
  Pfam: PF13560 — Helix-turn-helix domain (n=1)
  Pfam: PF00294 — pfkB family carbohydrate kinase (n=1)
  Pfam: PF13740 — ACT domain (n=1)
  InterPro: IPR000883 — Cytochrome c oxidase subunit I

COG0288 — -
  Pfam: PF12535 — UDP-X-like, N-terminal oligomerisation domain (n=1)
  Pfam: PF00583 — Acetyltransferase (GNAT) family (n=90)
  Pfam: PF04945 — YHS domain (n=12)
  InterPro: IPR001765 — Carbonic anhydrase

COG1622 — -
  Pfam: PF00127 

## 3. GO Terms and Pathway Assignments

Retrieve GO biological process terms and MetaCyc/KEGG pathway assignments
for gene clusters carrying novel OGs.

In [6]:
CACHE_GO = f'{DATA}/novel_og_go_terms.csv'

if os.path.exists(CACHE_GO) and os.path.getsize(CACHE_GO) > 100:
    og_go = pd.read_csv(CACHE_GO)
    print(f'Loaded cached GO terms: {len(og_go)} rows')
else:
    og_list = novel_ogs['og_id'].tolist()
    og_sql = "','".join(og_list)
    
    og_go = spark.sql(f"""
        SELECT SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] AS og_id,
               igo.go_id,
               igo.go_source,
               SUM(igo.n_supporting_analyses) AS total_support
        FROM kbase_ke_pangenome.eggnog_mapper_annotations ann
        JOIN kbase_ke_pangenome.interproscan_go igo
             ON ann.query_name = igo.gene_cluster_id
        WHERE ann.eggNOG_OGs IS NOT NULL
          AND SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] IN ('{og_sql}')
        GROUP BY 1, 2, 3
    """).toPandas()
    
    og_go.to_csv(CACHE_GO, index=False)
    print(f'Retrieved GO terms: {len(og_go)} OG-GO pairs across {og_go["og_id"].nunique()} OGs')

print(f'OGs with GO terms: {og_go["og_id"].nunique()}/50')
print(f'Unique GO terms: {og_go["go_id"].nunique()}')

Loaded cached GO terms: 1544 rows
OGs with GO terms: 48/50
Unique GO terms: 699


In [7]:
CACHE_PATH = f'{DATA}/novel_og_pathways.csv'

if os.path.exists(CACHE_PATH) and os.path.getsize(CACHE_PATH) > 100:
    og_pathways = pd.read_csv(CACHE_PATH)
    print(f'Loaded cached pathways: {len(og_pathways)} rows')
else:
    og_list = novel_ogs['og_id'].tolist()
    og_sql = "','".join(og_list)
    
    og_pathways = spark.sql(f"""
        SELECT SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] AS og_id,
               ipw.pathway_db,
               ipw.pathway_id,
               SUM(ipw.n_supporting_analyses) AS total_support
        FROM kbase_ke_pangenome.eggnog_mapper_annotations ann
        JOIN kbase_ke_pangenome.interproscan_pathways ipw
             ON ann.query_name = ipw.gene_cluster_id
        WHERE ann.eggNOG_OGs IS NOT NULL
          AND SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] IN ('{og_sql}')
        GROUP BY 1, 2, 3
    """).toPandas()
    
    og_pathways.to_csv(CACHE_PATH, index=False)
    print(f'Retrieved pathways: {len(og_pathways)} OG-pathway pairs across {og_pathways["og_id"].nunique()} OGs')

print(f'OGs with pathway assignments: {og_pathways["og_id"].nunique()}/50')
print(f'\nPathway database breakdown:')
print(og_pathways.groupby('pathway_db')['og_id'].nunique())

Loaded cached pathways: 2317 rows
OGs with pathway assignments: 39/50

Pathway database breakdown:
pathway_db
MetaCyc    39
Name: og_id, dtype: int64


## 4. Bakta Product Annotations

Cross-reference with Bakta reannotations for gene names and product descriptions
that may differ from eggNOG.

In [8]:
CACHE_BAKTA = f'{DATA}/novel_og_bakta.csv'

if os.path.exists(CACHE_BAKTA) and os.path.getsize(CACHE_BAKTA) > 100:
    og_bakta = pd.read_csv(CACHE_BAKTA)
    print(f'Loaded cached Bakta annotations: {len(og_bakta)} rows')
else:
    og_list = novel_ogs['og_id'].tolist()
    og_sql = "','".join(og_list)
    
    og_bakta = spark.sql(f"""
        SELECT SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] AS og_id,
               ba.gene,
               ba.product,
               ba.hypothetical,
               COUNT(*) AS n_clusters
        FROM kbase_ke_pangenome.eggnog_mapper_annotations ann
        JOIN kbase_ke_pangenome.bakta_annotations ba
             ON ann.query_name = ba.gene_cluster_id
        WHERE ann.eggNOG_OGs IS NOT NULL
          AND SPLIT(SPLIT(ann.eggNOG_OGs, ',')[0], '@')[0] IN ('{og_sql}')
        GROUP BY 1, 2, 3, 4
    """).toPandas()
    
    og_bakta.to_csv(CACHE_BAKTA, index=False)
    print(f'Retrieved Bakta annotations: {len(og_bakta)} rows across {og_bakta["og_id"].nunique()} OGs')

# What fraction are hypothetical?
hyp_frac = og_bakta.groupby('og_id').apply(
    lambda x: (x['hypothetical'] == True).sum() / len(x) if len(x) > 0 else 0
).reset_index(name='frac_hypothetical')

og_summary = og_summary.merge(hyp_frac, on='og_id', how='left')

# Get most common non-hypothetical bakta product per OG
bakta_named = og_bakta[(og_bakta['hypothetical'] != True) & (og_bakta['product'].notna())]
bakta_top = (bakta_named
    .sort_values('n_clusters', ascending=False)
    .groupby('og_id')
    .first()
    .reset_index()[['og_id', 'gene', 'product']]
    .rename(columns={'gene': 'bakta_gene', 'product': 'bakta_product'}))

og_summary = og_summary.merge(bakta_top, on='og_id', how='left')

print(f'\nFraction hypothetical by OG:')
print(f'  Fully annotated (0% hypothetical): {(hyp_frac["frac_hypothetical"] == 0).sum()}')
print(f'  Partially hypothetical: {((hyp_frac["frac_hypothetical"] > 0) & (hyp_frac["frac_hypothetical"] < 1)).sum()}')
print(f'  Fully hypothetical (100%): {(hyp_frac["frac_hypothetical"] == 1).sum()}')

Loaded cached Bakta annotations: 12580 rows

Fraction hypothetical by OG:
  Fully annotated (0% hypothetical): 0
  Partially hypothetical: 50
  Fully hypothetical (100%): 0


## 5. MGnify Cross-Reference

Check whether any novel OGs correspond to KEGG modules or COG categories
that are enriched in MGnify rhizosphere vs. soil species, and whether any
overlap with biosynthetic gene clusters.

In [9]:
# MGnify uses COG categories and KEGG modules at the genome level,
# not individual COG IDs. We can check if the COG functional categories
# of our novel OGs are enriched in rhizosphere vs soil.

CACHE_MGNIFY_COG = f'{DATA}/mgnify_rhizo_vs_soil_cog.csv'

if os.path.exists(CACHE_MGNIFY_COG) and os.path.getsize(CACHE_MGNIFY_COG) > 100:
    mgnify_cog = pd.read_csv(CACHE_MGNIFY_COG)
    print(f'Loaded cached MGnify COG profiles: {len(mgnify_cog)} rows')
else:
    # Get COG category profiles for rhizosphere vs soil biomes
    # genome_cog columns: genome_id, biome_id, cog_category, gene_count
    mgnify_cog = spark.sql("""
        SELECT c.biome_id, c.cog_category, 
               SUM(c.gene_count) AS total_genes,
               COUNT(DISTINCT c.genome_id) AS n_genomes
        FROM kescience_mgnify.genome_cog c
        WHERE c.biome_id IN ('tomato-rhizosphere', 'maize-rhizosphere', 
                             'barley-rhizosphere', 'soil')
        GROUP BY c.biome_id, c.cog_category
    """).toPandas()
    
    mgnify_cog.to_csv(CACHE_MGNIFY_COG, index=False)
    print(f'Retrieved MGnify COG profiles: {len(mgnify_cog)} rows')

# Check which COG categories from our novel OGs are present in MGnify
novel_cog_cats = og_summary[['og_id', 'cog_category']].dropna(subset=['cog_category'])
print(f'\nNovel OG COG categories: {novel_cog_cats["cog_category"].value_counts().to_dict()}')
print(f'\nMGnify biomes with COG data:')
print(mgnify_cog.groupby('biome_id')['n_genomes'].max())

Loaded cached MGnify COG profiles: 96 rows

Novel OG COG categories: {'S': 19, 'C': 6, 'P': 6, 'G': 4, 'Q': 3, 'O': 2, 'L': 2, 'E': 2, 'I': 2, 'CH': 1, 'F': 1, 'H': 1, 'K': 1}

MGnify biomes with COG data:
biome_id
barley-rhizosphere       82
maize-rhizosphere       297
soil                  16512
tomato-rhizosphere      517
Name: n_genomes, dtype: int64


In [10]:
# Check if any novel OGs have KEGG modules that appear in MGnify rhizosphere genomes
novel_kegg = og_summary[og_summary['kegg_module'].notna() & (og_summary['kegg_module'] != '')]
print(f'Novel OGs with KEGG module assignments: {len(novel_kegg)}')

if len(novel_kegg) > 0:
    # Get KEGG module data from MGnify for rhizosphere biomes
    # genome_kegg_module columns: genome_id, biome_id, kegg_module_id, gene_count
    kegg_modules = novel_kegg['kegg_module'].str.split(',').explode().str.strip().unique().tolist()
    km_sql = "','".join(kegg_modules)
    
    mgnify_kegg = spark.sql(f"""
        SELECT km.biome_id, km.kegg_module_id,
               SUM(km.gene_count) AS total_genes,
               COUNT(DISTINCT km.genome_id) AS n_genomes
        FROM kescience_mgnify.genome_kegg_module km
        WHERE km.biome_id IN ('tomato-rhizosphere', 'maize-rhizosphere', 
                              'barley-rhizosphere', 'soil')
          AND km.kegg_module_id IN ('{km_sql}')
        GROUP BY km.biome_id, km.kegg_module_id
    """).toPandas()
    
    print(f'\nKEGG modules from novel OGs found in MGnify rhizosphere:')
    if len(mgnify_kegg) > 0:
        print(mgnify_kegg.pivot_table(index='kegg_module_id', columns='biome_id', 
                                       values='n_genomes', fill_value=0))
    else:
        print('  None found')
else:
    print('No novel OGs have KEGG module assignments — skipping MGnify KEGG cross-reference')

Novel OGs with KEGG module assignments: 50



KEGG modules from novel OGs found in MGnify rhizosphere:
biome_id        barley-rhizosphere  maize-rhizosphere     soil  \
kegg_module_id                                                   
M00004                        83.0              335.0  16417.0   
M00006                        73.0              288.0  14482.0   
M00008                        75.0              292.0  14454.0   
M00048                        82.0              326.0  16076.0   
M00124                        77.0              301.0  13153.0   
M00154                        79.0              294.0  14350.0   
M00155                        73.0              288.0  14830.0   
M00546                        54.0              219.0  11226.0   

biome_id        tomato-rhizosphere  
kegg_module_id                      
M00004                       569.0  
M00006                       489.0  
M00008                       496.0  
M00048                       552.0  
M00124                       529.0  
M00154                

In [11]:
# Check MGnify BGC data — are any soil BGC species also in our pangenome plant-associated set?
CACHE_MGNIFY_BGC = f'{DATA}/mgnify_soil_bgc_summary.csv'

if os.path.exists(CACHE_MGNIFY_BGC) and os.path.getsize(CACHE_MGNIFY_BGC) > 100:
    mgnify_bgc = pd.read_csv(CACHE_MGNIFY_BGC)
    print(f'Loaded cached MGnify BGC summary: {len(mgnify_bgc)} rows')
else:
    # gene_bgc columns: bgc_id, genome_id, biome_id, contig_id, start_pos, end_pos,
    #                    nearest_mibig, nearest_mibig_class, score, partial
    mgnify_bgc = spark.sql("""
        SELECT s.gtdb_genus, s.gtdb_species,
               bgc.nearest_mibig_class,
               COUNT(*) AS n_bgcs,
               AVG(bgc.score) AS avg_score
        FROM kescience_mgnify.gene_bgc bgc
        JOIN kescience_mgnify.genome g ON bgc.genome_id = g.genome_id AND bgc.biome_id = g.biome_id
        JOIN kescience_mgnify.species s ON g.species_id = s.species_id AND g.biome_id = s.biome_id
        WHERE bgc.biome_id = 'soil'
        GROUP BY s.gtdb_genus, s.gtdb_species, bgc.nearest_mibig_class
    """).toPandas()
    
    mgnify_bgc.to_csv(CACHE_MGNIFY_BGC, index=False)
    print(f'Retrieved MGnify BGC summary: {len(mgnify_bgc)} genus-BGC pairs')

# Load our plant-associated species
sp_comp = pd.read_csv(f'{DATA}/species_compartment.csv')
plant_genera = sp_comp[sp_comp['is_plant_associated'] == True]['genus'].unique()

# Which plant-associated genera have BGCs in MGnify soil?
bgc_genera = mgnify_bgc['gtdb_genus'].unique()
overlap = set(plant_genera) & set(bgc_genera)
print(f'\nPlant-associated genera with BGCs in MGnify soil: {len(overlap)}')

if len(overlap) > 0:
    plant_bgc = mgnify_bgc[mgnify_bgc['gtdb_genus'].isin(overlap)]
    bgc_profile = plant_bgc.groupby('nearest_mibig_class')['n_bgcs'].sum().sort_values(ascending=False)
    print(f'\nBGC class distribution in plant-associated genera:')
    print(bgc_profile.head(10))

Retrieved MGnify BGC summary: 13584 genus-BGC pairs

Plant-associated genera with BGCs in MGnify soil: 0


## 6. Synthesis — Functional Classification of Novel OGs

Classify each novel OG into functional categories based on all annotation layers.

In [12]:
# COG category descriptions
COG_CATS = {
    'J': 'Translation, ribosomal structure',
    'A': 'RNA processing and modification',
    'K': 'Transcription',
    'L': 'Replication, recombination and repair',
    'B': 'Chromatin structure',
    'D': 'Cell cycle, division',
    'Y': 'Nuclear structure',
    'V': 'Defense mechanisms',
    'T': 'Signal transduction',
    'M': 'Cell wall/membrane/envelope biogenesis',
    'N': 'Cell motility',
    'Z': 'Cytoskeleton',
    'W': 'Extracellular structures',
    'U': 'Intracellular trafficking, secretion',
    'O': 'Post-translational modification, chaperones',
    'X': 'Mobilome: prophages, transposons',
    'C': 'Energy production and conversion',
    'G': 'Carbohydrate transport and metabolism',
    'E': 'Amino acid transport and metabolism',
    'F': 'Nucleotide transport and metabolism',
    'H': 'Coenzyme transport and metabolism',
    'I': 'Lipid transport and metabolism',
    'P': 'Inorganic ion transport and metabolism',
    'Q': 'Secondary metabolite biosynthesis',
    'R': 'General function prediction only',
    'S': 'Function unknown',
}

og_summary['cog_category_desc'] = og_summary['cog_category'].map(
    lambda x: COG_CATS.get(str(x)[0], 'Unknown') if pd.notna(x) and len(str(x)) > 0 else 'Unknown'
)

# Assign a plant-relevance hypothesis based on all annotation layers
def assign_hypothesis(row):
    """Generate a mechanistic hypothesis based on annotation layers."""
    signals = []
    desc = str(row.get('description', '')).lower()
    bakta = str(row.get('bakta_product', '')).lower()
    cat = str(row.get('cog_category', ''))
    domain = str(row.get('top_domain_desc', '')).lower()
    
    # Check for known plant-interaction keywords
    plant_kw = ['secretion', 'effector', 'flagell', 'chemotax', 'pilus', 'adhesin',
                'siderophore', 'iron', 'phosphat', 'nitrogen', 'nodulat', 'cellulose',
                'pectin', 'auxin', 'ethylene', 'biofilm', 'exopolysaccharide']
    transport_kw = ['transport', 'permease', 'ABC', 'porin', 'efflux', 'uptake']
    signal_kw = ['sensor', 'kinase', 'response regulator', 'two-component', 'signal']
    metabolism_kw = ['dehydrogenase', 'reductase', 'transferase', 'synthase', 'oxidase',
                     'hydrolase', 'lyase', 'isomerase', 'ligase']
    
    for kw in plant_kw:
        if kw in desc or kw in bakta or kw in domain:
            signals.append(f'Plant-interaction ({kw})')
    for kw in transport_kw:
        if kw in desc or kw in bakta or kw in domain:
            signals.append(f'Transport ({kw})')
    for kw in signal_kw:
        if kw in desc or kw in bakta or kw in domain:
            signals.append(f'Signaling ({kw})')
    
    if cat in ['T']:
        signals.append('Signal transduction (COG cat T)')
    elif cat in ['M', 'W']:
        signals.append('Cell envelope/surface (COG cat M/W)')
    elif cat in ['N']:
        signals.append('Motility (COG cat N)')
    elif cat in ['U']:
        signals.append('Secretion (COG cat U)')
    elif cat in ['P']:
        signals.append('Ion transport (COG cat P)')
    elif cat in ['Q']:
        signals.append('Secondary metabolites (COG cat Q)')
    elif cat in ['G']:
        signals.append('Carbohydrate metabolism (COG cat G)')
    
    hyp = row.get('frac_hypothetical', 1.0)
    if hyp > 0.8:
        signals.append('Largely uncharacterized')
    
    if not signals:
        return 'Function unknown — candidate for experimental characterization'
    return '; '.join(signals[:3])

og_summary['plant_relevance_hypothesis'] = og_summary.apply(assign_hypothesis, axis=1)

print('=== Functional Classification of 50 Novel Plant-Enriched OGs ===\n')
print(f'COG category distribution:')
print(og_summary['cog_category_desc'].value_counts())
print(f'\nHypothetical fraction: mean={og_summary["frac_hypothetical"].mean():.1%}, '
      f'fully hypothetical={og_summary["frac_hypothetical"].eq(1.0).sum()}')

=== Functional Classification of 50 Novel Plant-Enriched OGs ===

COG category distribution:
cog_category_desc
Function unknown                               19
Energy production and conversion                7
Inorganic ion transport and metabolism          6
Carbohydrate transport and metabolism           4
Secondary metabolite biosynthesis               3
Post-translational modification, chaperones     2
Replication, recombination and repair           2
Amino acid transport and metabolism             2
Lipid transport and metabolism                  2
Nucleotide transport and metabolism             1
Coenzyme transport and metabolism               1
Transcription                                   1
Name: count, dtype: int64

Hypothetical fraction: mean=1.2%, fully hypothetical=0


In [13]:
# Print the full annotation table
print('\n=== Complete Novel OG Annotation Table ===\n')
print(f'{"OG":<10} {"OR":>5} {"phyloOR":>7} {"Plant%":>7} {"Name":<15} {"COG":>4} {"Hyp%":>5} {"Description":<50}')
print('-' * 110)
for _, row in og_summary.iterrows():
    name = str(row['preferred_name'])[:14] if pd.notna(row['preferred_name']) and row['preferred_name'] != '' else '-'
    desc = str(row['description'])[:49] if pd.notna(row['description']) and row['description'] != '' else '-'
    cat = str(row['cog_category'])[:4] if pd.notna(row['cog_category']) else '-'
    hyp = f"{row['frac_hypothetical']:.0%}" if pd.notna(row.get('frac_hypothetical')) else '-'
    print(f"{row['og_id']:<10} {row['odds_ratio']:>5.1f} {row['or_phylo_controlled']:>7.1f} "
          f"{row['prev_plant']:>6.1%} {name:<15} {cat:>4} {hyp:>5} {desc:<50}")


=== Complete Novel OG Annotation Table ===

OG            OR phyloOR  Plant% Name             COG  Hyp% Description                                       
--------------------------------------------------------------------------------------------------------------
COG1845     14.5     8.7  93.2% ctaE               C    0% Cytochrome c oxidase subunit III                  
COG0316     13.9     6.3  94.3% iscA               S    0% Belongs to the HesB IscA family                   
COG0843     13.7     7.0  93.1% ctaD               C    1% Cytochrome c oxidase is the component of the resp 
COG0288     13.5     7.9  94.5% -                  P    1% Reversible hydration of carbon dioxide            
COG1622     13.0     7.9  92.6% -                  C    1% Subunits I and II form the functional core of the 
COG0109     13.0     6.7  92.9% ctaB               O    4% Converts heme B (protoheme IX) to heme O by subst 
COG0654     12.7     8.3  91.2% -                 CH    0% FAD binding do

In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# Panel A: COG category distribution
cat_counts = og_summary['cog_category_desc'].value_counts()
axes[0].barh(range(len(cat_counts)), cat_counts.values, color='steelblue')
axes[0].set_yticks(range(len(cat_counts)))
axes[0].set_yticklabels(cat_counts.index, fontsize=9)
axes[0].set_xlabel('Number of OGs')
axes[0].set_title('A) COG Functional Categories')
axes[0].invert_yaxis()

# Panel B: Odds ratio vs hypothetical fraction
ax1 = axes[1]
sc = ax1.scatter(og_summary['or_phylo_controlled'], 
                  og_summary['frac_hypothetical'],
                  s=og_summary['prev_plant'] * 200, 
                  alpha=0.6, c='coral', edgecolors='black', linewidth=0.5)
for _, row in og_summary.head(5).iterrows():
    name = row['preferred_name'] if pd.notna(row['preferred_name']) and row['preferred_name'] != '' else row['og_id']
    ax1.annotate(str(name)[:12], (row['or_phylo_controlled'], row['frac_hypothetical']),
                 fontsize=7, ha='left', va='bottom')
ax1.set_xlabel('Phylo-controlled OR')
ax1.set_ylabel('Fraction hypothetical')
ax1.set_title('B) Enrichment vs Characterization')
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# Panel C: Core fraction distribution
axes[2].hist(og_summary['pct_core'], bins=20, color='seagreen', edgecolor='black', alpha=0.7)
axes[2].axvline(x=0.468, color='red', linestyle='--', label='Genome-wide baseline (46.8%)')
axes[2].set_xlabel('Core fraction')
axes[2].set_ylabel('Number of OGs')
axes[2].set_title('C) Genomic Architecture of Novel OGs')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIG}/novel_og_annotation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG}/novel_og_annotation.png')

Saved: ../figures/novel_og_annotation.png


In [15]:
# Save final annotation table
save_cols = ['og_id', 'odds_ratio', 'or_phylo_controlled', 'q_value',
             'prev_plant', 'prev_nonplant', 'n_clusters', 'n_species', 'pct_core',
             'preferred_name', 'description', 'cog_category', 'cog_category_desc',
             'kegg_ko', 'kegg_module', 'kegg_pathway', 'ec_number', 'go_terms', 'pfams',
             'top_domain_db', 'top_domain_acc', 'top_domain_desc', 'interpro_acc', 'interpro_desc',
             'bakta_gene', 'bakta_product', 'frac_hypothetical',
             'plant_relevance_hypothesis']
# Only include columns that exist
save_cols = [c for c in save_cols if c in og_summary.columns]
og_summary[save_cols].to_csv(f'{DATA}/novel_og_annotations.csv', index=False)
print(f'Saved: {DATA}/novel_og_annotations.csv ({len(og_summary)} OGs)')

# Save domain details
og_domains.to_csv(f'{DATA}/novel_og_domains.csv', index=False)
print(f'Saved: {DATA}/novel_og_domains.csv ({len(og_domains)} domain hits)')

Saved: ../data/novel_og_annotations.csv (50 OGs)
Saved: ../data/novel_og_domains.csv (4455 domain hits)


In [16]:
# Final summary
print('\n' + '='*80)
print('NB09 SUMMARY: Novel OG Functional Annotation')
print('='*80)

n_annotated = og_summary[og_summary['description'].notna() & (og_summary['description'] != '')].shape[0]
n_named = og_summary[og_summary['preferred_name'].notna() & (og_summary['preferred_name'] != '')].shape[0]
n_with_domains = og_summary[og_summary['top_domain_acc'].notna()].shape[0] if 'top_domain_acc' in og_summary.columns else 0
n_with_kegg = og_summary[og_summary['kegg_ko'].notna() & (og_summary['kegg_ko'] != '')].shape[0]
n_low_hyp = og_summary[og_summary['frac_hypothetical'] < 0.5].shape[0] if 'frac_hypothetical' in og_summary.columns else 0

print(f'\nAnnotation coverage of 50 novel plant-enriched OGs:')
print(f'  With eggNOG description: {n_annotated}/50')
print(f'  With gene name: {n_named}/50')
print(f'  With InterProScan domains: {n_with_domains}/50')
print(f'  With KEGG orthology: {n_with_kegg}/50')
print(f'  Well-characterized (<50% hypothetical): {n_low_hyp}/50')

print(f'\nTop 10 OGs with mechanistic hypotheses:')
for _, row in og_summary.head(10).iterrows():
    print(f"  {row['og_id']}: {row['plant_relevance_hypothesis']}")

print(f'\nOutputs:')
print(f'  data/novel_og_annotations.csv — full annotation table')
print(f'  data/novel_og_domains.csv — InterProScan domain details')
print(f'  figures/novel_og_annotation.png — 3-panel annotation overview')


NB09 SUMMARY: Novel OG Functional Annotation

Annotation coverage of 50 novel plant-enriched OGs:
  With eggNOG description: 50/50
  With gene name: 50/50
  With InterProScan domains: 50/50
  With KEGG orthology: 50/50
  Well-characterized (<50% hypothetical): 50/50

Top 10 OGs with mechanistic hypotheses:
  COG1845: Function unknown — candidate for experimental characterization
  COG0316: Plant-interaction (iron)
  COG0843: Function unknown — candidate for experimental characterization
  COG0288: Ion transport (COG cat P)
  COG1622: Function unknown — candidate for experimental characterization
  COG0109: Function unknown — candidate for experimental characterization
  COG0654: Function unknown — candidate for experimental characterization
  COG2010: Function unknown — candidate for experimental characterization
  COG3832: Function unknown — candidate for experimental characterization
  COG0861: Ion transport (COG cat P)

Outputs:
  data/novel_og_annotations.csv — full annotation tab